In [1]:
# import required libraries
# Data Handling
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Data Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

# Feature Selection
from sklearn.feature_selection import SelectKBest, chi2, RFE

# ML Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

# Imbalanced Data
from imblearn.over_sampling import SMOTE

# Warnings
import warnings
warnings.filterwarnings("ignore")

In [2]:
#! pip install imblearn

In [3]:
# load the dataset
raw_df = pd.read_csv("Patient_Health_Records_Raw.csv")

In [4]:
# 1. Check dataset shape
raw_df.shape

(10000, 17)

In [5]:
# 2. Check first rows
raw_df.head()

,Patient_ID,Name,Age,Gender,City,BMI,Blood_Pressure,Heart_Rate,Cholesterol_Level,Diabetic,Smoker,Medications,Last_Visit_Date,Follow_Up,Diagnosis_Code,Notes,Has_Disease
0,ID-00001,"SMITH, ALICE",250,Unknown,Boston,22kg/m2,120 over 80,83,190,Unknown,yes,NaN,Apr 15 2021,30,A00,Patient feeling fine 🙂,0
1,ID-00002,"SMITH, ALICE",-5,FEMALE,newyork,25.8,NaN,500,normal,N,No,NaN,2021-07-10,NaN,unknown,"BP high, needs recheck",unknown
2,ID-00003,John Doe,-5,NaN,Nwe Yrok,22kg/m2,120 - 80,500,190,no,Former,"aspirin,metformin",Apr 06 2021,30,B20,NaN,unknown
3,ID-00004,michael brown,80,F,New York,33.4,NaN,eighty,250,y,EX-smoker,NaN,20241113,30,NaN,Follow up soon,unknown
4,ID-00005,Jane Doe,250,male,la,30.1,120 over 80,0,normal,no,No,NaN,2025-09-27,NaN,I10,Patient feeling fine 🙂,1


In [6]:
# 3. Check column names
raw_df.columns

Index(['Patient_ID', 'Name', 'Age', 'Gender', 'City', 'BMI', 'Blood_Pressure',
       'Heart_Rate', 'Cholesterol_Level', 'Diabetic', 'Smoker', 'Medications',
       'Last_Visit_Date', 'Follow_Up', 'Diagnosis_Code', 'Notes',
       'Has_Disease'],
      dtype='object')

# Drop unwanted columns which is not for analysis

In [7]:
# Drop unwanted columns
raw_df = raw_df.drop(columns=["Patient_ID", "Name", ])

In [8]:
# 4. Check data types
raw_df.dtypes

Age                  object
Gender               object
City                 object
BMI                  object
Blood_Pressure       object
Heart_Rate           object
Cholesterol_Level    object
Diabetic             object
Smoker               object
Medications          object
Last_Visit_Date      object
Follow_Up            object
Diagnosis_Code       object
Notes                object
Has_Disease          object
dtype: object

In [9]:
raw_df["Has_Disease"].value_counts(dropna=False)

Has_Disease
NaN        2520
0          2513
unknown    2506
1          2461
Name: count, dtype: int64

In [10]:
(raw_df["Has_Disease"].astype(str).str.lower() == "unknown").sum()

np.int64(2506)

In [11]:
# Check original Cholesterol_Level values

raw_df["Cholesterol_Level"].unique()

array(['190', 'normal', '250', 'high', nan], dtype=object)

In [12]:
# Count each value including missing values

raw_df["Cholesterol_Level"].value_counts(dropna=False)

Cholesterol_Level
NaN       2052
190       2048
normal    1994
250       1970
high      1936
Name: count, dtype: int64

In [13]:
# 6. Check missing values
raw_df.isnull().sum()

Age                     0
Gender               1707
City                 1003
BMI                  2418
Blood_Pressure       2030
Heart_Rate              0
Cholesterol_Level    2052
Diabetic             1641
Smoker               3234
Medications          4024
Last_Visit_Date         0
Follow_Up            2524
Diagnosis_Code       1675
Notes                4024
Has_Disease          2520
dtype: int64

In [14]:
# Remove NaN and unknown values from Has_Disease
raw_df = raw_df[
    (raw_df["Has_Disease"].astype(str).str.lower() != "unknown") &
    (raw_df["Has_Disease"].notna())
].reset_index(drop=True)

# Check the new index
print(raw_df.index)

RangeIndex(start=0, stop=4974, step=1)


In [16]:
# 1. Check dataset shape
raw_df.shape

(4974, 15)

In [17]:
# 6. Check missing values
raw_df.isnull().sum()

Age                     0
Gender                830
City                  540
BMI                  1205
Blood_Pressure       1032
Heart_Rate              0
Cholesterol_Level    1031
Diabetic              821
Smoker               1598
Medications          1986
Last_Visit_Date         0
Follow_Up            1272
Diagnosis_Code        864
Notes                2046
Has_Disease             0
dtype: int64

In [18]:
# Count each value including missing values

raw_df["Cholesterol_Level"].value_counts(dropna=False)

Cholesterol_Level
NaN       1031
190       1024
normal    1016
250        974
high       929
Name: count, dtype: int64

# Splitting cholestral level in to two columns

In [20]:
# Create numeric cholesterol column
raw_df["Cholesterol_Value"] = pd.to_numeric(
    raw_df["Cholesterol_Level"], errors="coerce"
).astype("Int64")

# Create categorical cholesterol column
raw_df["Cholesterol_Status"] = (
    raw_df["Cholesterol_Level"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Keep only valid categorical values
raw_df["Cholesterol_Status"] = raw_df["Cholesterol_Status"].where(
    raw_df["Cholesterol_Status"].isin(["normal", "high"]),
    pd.NA
)

# Type casting

In [21]:
# ============================================================
# Convert columns to their appropriate data types
# ============================================================

# Age → nullable integer
raw_df["Age"] = pd.to_numeric(
    raw_df["Age"], errors="coerce"
).astype("Int64")


# BMI → nullable float
raw_df["BMI"] = pd.to_numeric(
    raw_df["BMI"], errors="coerce"
).astype("Float64")


# Heart Rate → nullable integer
raw_df["Heart_Rate"] = pd.to_numeric(
    raw_df["Heart_Rate"], errors="coerce"
).astype("Int64")


# Cholesterol_Value → nullable integer
raw_df["Cholesterol_Value"] = pd.to_numeric(
    raw_df["Cholesterol_Value"], errors="coerce"
).astype("Int64")


# Cholesterol_Status → string
raw_df["Cholesterol_Status"] = (
    raw_df["Cholesterol_Status"].astype("string")
)


# Follow Up → nullable integer
raw_df["Follow_Up"] = pd.to_numeric(
    raw_df["Follow_Up"], errors="coerce"
).astype("Int64")


# Blood Pressure → string
raw_df["Blood_Pressure"] = (
    raw_df["Blood_Pressure"]
    .astype("string")
    .str.strip()
)


# Last Visit Date → datetime
raw_df["Last_Visit_Date"] = pd.to_datetime(
    raw_df["Last_Visit_Date"], errors="coerce"
)


# Has_Disease → nullable integer
raw_df["Has_Disease"] = pd.to_numeric(
    raw_df["Has_Disease"], errors="coerce"
).astype("Int64")


# ============================================================
# Convert categorical columns to object
# ============================================================

categorical_cols = [
    "Gender",
    "City",
    "Blood_Pressure",
    "Diabetic",
    "Smoker",
    "Medications",
    "Diagnosis_Code",
    "Notes",
    "Cholesterol_Status"
]

raw_df[categorical_cols] = raw_df[categorical_cols].astype("object")




In [22]:
# ============================================================
# Check final data types
# ============================================================

print(raw_df.dtypes)

Age                            Int64
Gender                        object
City                          object
BMI                          Float64
Blood_Pressure                object
Heart_Rate                     Int64
Cholesterol_Level             object
Diabetic                      object
Smoker                        object
Medications                   object
Last_Visit_Date       datetime64[ns]
Follow_Up                      Int64
Diagnosis_Code                object
Notes                         object
Has_Disease                    Int64
Cholesterol_Value              Int64
Cholesterol_Status            object
dtype: object


In [23]:
raw_df.head()

,Age,Gender,City,BMI,Blood_Pressure,Heart_Rate,Cholesterol_Level,Diabetic,Smoker,Medications,Last_Visit_Date,Follow_Up,Diagnosis_Code,Notes,Has_Disease,Cholesterol_Value,Cholesterol_Status
0,250,Unknown,Boston,<NA>,120 over 80,83,190,Unknown,yes,NaN,2021-04-15,30,A00,Patient feeling fine 🙂,0,190,<NA>
1,250,male,la,30.1,120 over 80,0,normal,no,No,NaN,NaT,<NA>,I10,Patient feeling fine 🙂,1,<NA>,normal
2,-5,Unknown,NaN,<NA>,120 - 80,63,normal,no,EX-smoker,NaN,NaT,30,E11.9,NaN,0,<NA>,normal
3,250,NaN,Boston,<NA>,120 over 80,500,high,N,Former,Aspirin; Metformin,NaT,30,unknown,NaN,0,<NA>,high
4,-5,Unknown,Chicago,21.5,<NA>,<NA>,NaN,no,Former,NaN,NaT,30,A00,"BP high, needs recheck",1,<NA>,<NA>


In [24]:
# 5. Check dataset information
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4974 entries, 0 to 4973
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Age                 3746 non-null   Int64         
 1   Gender              4144 non-null   object        
 2   City                4434 non-null   object        
 3   BMI                 2545 non-null   Float64       
 4   Blood_Pressure      3942 non-null   object        
 5   Heart_Rate          3751 non-null   Int64         
 6   Cholesterol_Level   3943 non-null   object        
 7   Diabetic            4153 non-null   object        
 8   Smoker              3376 non-null   object        
 9   Medications         2988 non-null   object        
 10  Last_Visit_Date     1212 non-null   datetime64[ns]
 11  Follow_Up           2425 non-null   Int64         
 12  Diagnosis_Code      4110 non-null   object        
 13  Notes               2928 non-null   object      

In [25]:
# 6. Check missing values
raw_df.isnull().sum()

Age                   1228
Gender                 830
City                   540
BMI                   2429
Blood_Pressure        1032
Heart_Rate            1223
Cholesterol_Level     1031
Diabetic               821
Smoker                1598
Medications           1986
Last_Visit_Date       3762
Follow_Up             2549
Diagnosis_Code         864
Notes                 2046
Has_Disease              0
Cholesterol_Value     2976
Cholesterol_Status    3029
dtype: int64

In [26]:
raw_df["Has_Disease"].value_counts()

Has_Disease
0    2513
1    2461
Name: count, dtype: Int64

In [27]:
raw_df["Has_Disease"].value_counts(normalize=True) * 100

Has_Disease
0    50.522718
1    49.477282
Name: proportion, dtype: Float64

In [28]:
raw_df["Has_Disease"].value_counts(dropna=False)

Has_Disease
0    2513
1    2461
Name: count, dtype: Int64

In [29]:
(raw_df["Has_Disease"].astype(str).str.lower() == "unknown").sum()

np.int64(0)

In [30]:
# 6. Check missing values
raw_df.isnull().sum()

Age                   1228
Gender                 830
City                   540
BMI                   2429
Blood_Pressure        1032
Heart_Rate            1223
Cholesterol_Level     1031
Diabetic               821
Smoker                1598
Medications           1986
Last_Visit_Date       3762
Follow_Up             2549
Diagnosis_Code         864
Notes                 2046
Has_Disease              0
Cholesterol_Value     2976
Cholesterol_Status    3029
dtype: int64

In [31]:
# Remove unwanted spaces from all string/object columns
for col in raw_df.select_dtypes(include="object").columns:
    raw_df[col] = raw_df[col].str.strip()

In [32]:
# Fix invalid Age values
# Keep only realistic ages (0–120); invalid values become NaN

raw_df["Age"] = pd.to_numeric(raw_df["Age"], errors="coerce")
raw_df.loc[(raw_df["Age"] < 0) | (raw_df["Age"] > 120), "Age"] = np.nan

In [33]:
# Standardize Gender categories

raw_df["Gender"] = raw_df["Gender"].str.strip().str.lower()

raw_df["Gender"] = raw_df["Gender"].replace({
    "m": "Male",
    "male": "Male",
    "f": "Female",
    "female": "Female",
    "unknown": np.nan
})

In [34]:
# Standardize City names

raw_df["City"] = (
    raw_df["City"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Combine different city formats
raw_df["City"] = raw_df["City"].replace({
    "la": "Los Angeles",
    "los angeles": "Los Angeles",
    
    "ny": "New York",
    "new york": "New York",
    "newyork": "New York",
    "nwe yrok": "New York"
})

# Correct capitalization of remaining city names
raw_df["City"] = raw_df["City"].str.title()

In [35]:
# Clean and convert BMI to numeric

raw_df["BMI"] = (
    raw_df["BMI"]
    .astype("string")
    .str.replace("kg/m2", "", regex=False)
    .str.strip()
)

raw_df["BMI"] = pd.to_numeric(raw_df["BMI"], errors="coerce")

In [36]:
# Standardize Blood_Pressure format

raw_df["Blood_Pressure"] = (
    raw_df["Blood_Pressure"]
    .astype("string")
    .str.strip()
    .str.replace(" over ", "/", regex=False)
    .str.replace(" - ", "/", regex=False)
)

# Convert N/A and other invalid values to NaN
#raw_df["Blood_Pressure"] = raw_df["Blood_Pressure"].replace(
#    ["N/A", "NA", "n/a", "na", ""], np.nan)



# Convert common missing-value representations to np.nan
raw_df = raw_df.replace(
    ["N/A", "NA", "n/a", "na", ""],
    np.nan
)

# Split Blood_Pressure into two columns
raw_df[["Systolic_BP", "Diastolic_BP"]] = (
    raw_df["Blood_Pressure"]
    .str.split("/", n=1, expand=True)
)

# Convert both columns to numeric
raw_df["Systolic_BP"] = pd.to_numeric(
    raw_df["Systolic_BP"], errors="coerce"
)

raw_df["Diastolic_BP"] = pd.to_numeric(
    raw_df["Diastolic_BP"], errors="coerce"
)

# Remove the original Blood_Pressure column
#raw_df.drop("Blood_Pressure", axis=1, inplace=True)

In [37]:
# Clean and convert Heart_Rate to numeric

raw_df["Heart_Rate"] = pd.to_numeric(
    raw_df["Heart_Rate"], errors="coerce"
)

# Replace invalid Heart_Rate values with NaN

raw_df.loc[
    (raw_df["Heart_Rate"] <= 0) | (raw_df["Heart_Rate"] > 200),
    "Heart_Rate"
] = np.nan

In [38]:
# Standardize Diabetic categories

raw_df["Diabetic"] = (
    raw_df["Diabetic"]
    .astype("string")
    .str.strip()
    .str.lower()
)

raw_df["Diabetic"] = raw_df["Diabetic"].replace({
    "y": "Yes",
    "yes": "Yes",
    "n": "No",
    "no": "No",
    "unknown": np.nan
})

In [39]:
# Standardize Smoker categories

raw_df["Smoker"] = (
    raw_df["Smoker"]
    .astype("string")
    .str.strip()
    .str.lower()
)

raw_df["Smoker"] = raw_df["Smoker"].replace({
    "yes": "Yes",
    "no": "No",
    "former": "Former",
    "ex-smoker": "Former",
    "n/a": np.nan
})

In [40]:
# Standardize Medications

raw_df["Medications"] = (
    raw_df["Medications"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Standardize separators
raw_df["Medications"] = (
    raw_df["Medications"]
    .str.replace(",", ";", regex=False)
    .str.replace("/", ";", regex=False)
)

# Standardize spaces around ;
raw_df["Medications"] = (
    raw_df["Medications"]
    .str.replace(r"\s*;\s*", "; ", regex=True)
)

# Convert None / N/A to NaN
raw_df["Medications"] = raw_df["Medications"].replace(
    ["none", "n/a", "na", ""], np.nan
)

# Standardize medication names
raw_df["Medications"] = raw_df["Medications"].str.title()

In [41]:
# Clean and convert Last_Visit_Date to datetime

raw_df["Last_Visit_Date"] = (
    raw_df["Last_Visit_Date"]
    .astype("string")
    .str.strip()
)

# Convert different date formats
raw_df["Last_Visit_Date"] = pd.to_datetime(
    raw_df["Last_Visit_Date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

In [42]:
# Clean Follow_Up

raw_df["Follow_Up"] = (
    raw_df["Follow_Up"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Convert text value to number of days
raw_df["Follow_Up"] = raw_df["Follow_Up"].replace({
    "two weeks": "14"
})

# Convert to numeric
raw_df["Follow_Up"] = pd.to_numeric(
    raw_df["Follow_Up"], errors="coerce"
)

In [43]:
# Standardize Diagnosis_Code

raw_df["Diagnosis_Code"] = (
    raw_df["Diagnosis_Code"]
    .astype("string")
    .str.strip()
)

raw_df["Diagnosis_Code"] = raw_df["Diagnosis_Code"].replace({
    "unknown": np.nan,
    "Unknown": np.nan,
    "N/A": np.nan,
    "n/a": np.nan,
    "": np.nan
})

In [44]:
# Clean Notes

raw_df["Notes"] = (
    raw_df["Notes"]
    .astype("string")
    .str.strip()
)

# Convert missing-value indicators to NaN
raw_df["Notes"] = raw_df["Notes"].replace({
    "n/a": np.nan,
    "N/A": np.nan,
    "na": np.nan,
    "NA": np.nan,
    "": np.nan
})

In [45]:
# Clean Has_Disease target

raw_df["Has_Disease"] = (
    raw_df["Has_Disease"]
    .astype("string")
    .str.strip()
)

# Convert unknown and missing values to NaN
raw_df["Has_Disease"] = raw_df["Has_Disease"].replace({
    "unknown": np.nan,
    "Unknown": np.nan,
    "": np.nan,
    "N/A": np.nan,
    "n/a": np.nan
})

# Convert valid 0 and 1 to numeric
raw_df["Has_Disease"] = pd.to_numeric(
    raw_df["Has_Disease"], errors="coerce"
)

# Remove rows where the target is missing
raw_df = raw_df.dropna(subset=["Has_Disease"])

# Convert target to integer
raw_df["Has_Disease"] = raw_df["Has_Disease"].astype(int)

In [46]:
# Keep original Blood_Pressure column

# Convert Systolic BP to numeric
raw_df["Systolic_BP"] = pd.to_numeric(
    raw_df["Systolic_BP"], errors="coerce"
).astype("Int64")

# Convert Diastolic BP to numeric
raw_df["Diastolic_BP"] = pd.to_numeric(
    raw_df["Diastolic_BP"], errors="coerce"
).astype("Int64")

In [48]:
# Check data types

raw_df[["Systolic_BP", "Diastolic_BP"]].dtypes

Systolic_BP     Int64
Diastolic_BP    Int64
dtype: object

In [49]:
raw_df.head()

,Age,Gender,City,BMI,Blood_Pressure,Heart_Rate,Cholesterol_Level,Diabetic,Smoker,Medications,Last_Visit_Date,Follow_Up,Diagnosis_Code,Notes,Has_Disease,Cholesterol_Value,Cholesterol_Status,Systolic_BP,Diastolic_BP
0,<NA>,NaN,Boston,<NA>,120/80,83,190,<NA>,Yes,<NA>,2021-04-15,30,A00,Patient feeling fine 🙂,0,190,<NA>,120,80
1,<NA>,Male,Los Angeles,30.1,120/80,<NA>,normal,No,No,<NA>,NaT,<NA>,I10,Patient feeling fine 🙂,1,<NA>,normal,120,80
2,<NA>,NaN,<NA>,<NA>,120/80,63,normal,No,Former,<NA>,NaT,30,E11.9,<NA>,0,<NA>,normal,120,80
3,<NA>,NaN,Boston,<NA>,120/80,<NA>,high,No,Former,Aspirin; Metformin,NaT,30,<NA>,<NA>,0,<NA>,high,120,80
4,<NA>,NaN,Chicago,21.5,<NA>,<NA>,NaN,No,Former,<NA>,NaT,30,A00,"BP high, needs recheck",1,<NA>,<NA>,<NA>,<NA>


In [50]:
# Check value counts of Cholesterol_Status

raw_df["Cholesterol_Status"].value_counts(dropna=False)

Cholesterol_Status
<NA>      3029
normal    1016
high       929
Name: count, dtype: int64

In [51]:
raw_df["Cholesterol_Status"].unique()

array([<NA>, 'normal', 'high'], dtype=object)

In [52]:
# Check value counts of Cholesterol_Status

raw_df["Cholesterol_Value"].value_counts(dropna=False)

Cholesterol_Value
<NA>    2976
190     1024
250      974
Name: count, dtype: Int64

In [53]:
raw_df["Cholesterol_Value"].unique()

<IntegerArray>
[190, <NA>, 250]
Length: 3, dtype: Int64

In [54]:
raw_df.shape

(4974, 19)

In [55]:
# Check column names

raw_df.columns.tolist()

['Age',
 'Gender',
 'City',
 'BMI',
 'Blood_Pressure',
 'Heart_Rate',
 'Cholesterol_Level',
 'Diabetic',
 'Smoker',
 'Medications',
 'Last_Visit_Date',
 'Follow_Up',
 'Diagnosis_Code',
 'Notes',
 'Has_Disease',
 'Cholesterol_Value',
 'Cholesterol_Status',
 'Systolic_BP',
 'Diastolic_BP']

In [56]:
# Check data types

raw_df.dtypes

Age                            Int64
Gender                        object
City                  string[python]
BMI                          Float64
Blood_Pressure        string[python]
Heart_Rate                     Int64
Cholesterol_Level             object
Diabetic              string[python]
Smoker                string[python]
Medications           string[python]
Last_Visit_Date       datetime64[ns]
Follow_Up                      Int64
Diagnosis_Code        string[python]
Notes                 string[python]
Has_Disease                    int64
Cholesterol_Value              Int64
Cholesterol_Status            object
Systolic_BP                    Int64
Diastolic_BP                   Int64
dtype: object

In [57]:
# Convert all categorical columns to object

categorical_cols = [
    "Gender",
    "City",
    "Diabetic",
    "Smoker",
    "Medications",
    "Diagnosis_Code",
    "Notes",
    "Cholesterol_Status"
]

raw_df[categorical_cols] = raw_df[categorical_cols].astype("object")

In [58]:
# Check data types

raw_df.dtypes

Age                            Int64
Gender                        object
City                          object
BMI                          Float64
Blood_Pressure        string[python]
Heart_Rate                     Int64
Cholesterol_Level             object
Diabetic                      object
Smoker                        object
Medications                   object
Last_Visit_Date       datetime64[ns]
Follow_Up                      Int64
Diagnosis_Code                object
Notes                         object
Has_Disease                    int64
Cholesterol_Value              Int64
Cholesterol_Status            object
Systolic_BP                    Int64
Diastolic_BP                   Int64
dtype: object

In [59]:
# 7. Check duplicate rows
raw_df.duplicated().sum()

np.int64(3)

In [60]:
# Remove duplicate rows
raw_df = raw_df.drop_duplicates()

In [61]:
raw_df.duplicated().sum()

np.int64(0)

In [62]:
raw_df["Has_Disease"].value_counts()

Has_Disease
0    2512
1    2459
Name: count, dtype: int64

In [68]:
# Create a working copy of the cleaned dataset
df = raw_df.copy()

In [69]:
df.head()

,Age,Gender,City,BMI,Blood_Pressure,Heart_Rate,Cholesterol_Level,Diabetic,Smoker,Medications,Last_Visit_Date,Follow_Up,Diagnosis_Code,Notes,Has_Disease,Cholesterol_Value,Cholesterol_Status,Systolic_BP,Diastolic_BP
0,<NA>,NaN,Boston,<NA>,120/80,83,190,<NA>,Yes,<NA>,2021-04-15,30,A00,Patient feeling fine 🙂,0,190,<NA>,120,80
1,<NA>,Male,Los Angeles,30.1,120/80,<NA>,normal,No,No,<NA>,NaT,<NA>,I10,Patient feeling fine 🙂,1,<NA>,normal,120,80
2,<NA>,NaN,<NA>,<NA>,120/80,63,normal,No,Former,<NA>,NaT,30,E11.9,<NA>,0,<NA>,normal,120,80
3,<NA>,NaN,Boston,<NA>,120/80,<NA>,high,No,Former,Aspirin; Metformin,NaT,30,<NA>,<NA>,0,<NA>,high,120,80
4,<NA>,NaN,Chicago,21.5,<NA>,<NA>,NaN,No,Former,<NA>,NaT,30,A00,"BP high, needs recheck",1,<NA>,<NA>,<NA>,<NA>


In [74]:
df.dtypes

Age                            Int64
Gender                        object
City                          object
BMI                          Float64
Blood_Pressure        string[python]
Heart_Rate                     Int64
Cholesterol_Level             object
Diabetic                      object
Smoker                        object
Medications                   object
Last_Visit_Date       datetime64[ns]
Follow_Up                      Int64
Diagnosis_Code                object
Notes                         object
Has_Disease                    int64
Cholesterol_Value              Int64
Cholesterol_Status            object
Systolic_BP                    Int64
Diastolic_BP                   Int64
dtype: object

In [70]:
# Save the cleaned dataset as a CSV file
df.to_csv("Patient_Health_Records_Cleaned.csv", index=False)